# Tutorial 2.3 — Sea Surface Height and Ocean Currents

Warm water takes up more room than cold, so a column of ocean with a thick warm layer on top stands taller than a cold column beside it. Measure the sea surface precisely enough and you are measuring the heat stored underneath it. Away from the equator the water also runs along the slopes of that surface, so the same measurement tells you which way it is moving.

Part 1 maps **sea surface height anomalies (SSHA)** from SWOT. Part 2 maps **ocean surface currents** from OSCAR.

**The plan:** the SWOT data and why it is not a grid (§1–2) → map it and choose colour limits (§3) → every period on one scale (§4) → watch a month being collected (§5) → grid the samples and measure the basin's east–west tilt (§6) → December 2022, the one period SWOT cannot see (§7) → why nothing in orbit measures a current (§8) → the OSCAR data (§9–10) → map the currents (§11–12) → the equatorial current bands (§13).

**Required: sections 6, 12 and 13.** Questions 5 and 8 come from section 6, question 6 from sections 12 and 13. Section 5 is optional and not marked.

> ### ⚠️ The SWOT data is a point cloud, not a grid
>
> PACE and Himawari handed you a neat rectangular grid. SWOT arrives as a **list of individual measurements**, each carrying its own longitude and latitude. Section 2 explains what that changes. The OSCAR data in part 2 is back on a grid.

## Setting up

Run the cell below first, before anything else on this page.

Colab hands you a completely empty machine, so this cell installs the packages the tutorial needs and creates the folders the notebook writes into.

Unlike notebooks 2.1 and 2.2, nothing here needs a data provider login: the extracts are downloaded over plain HTTPS from the course bucket.

Colab forgets everything between sessions, so run it again whenever you reopen the notebook.

In [ ]:
# --- Setup: run this cell first. Safe to re-run. ---------------------------
# Colab gives you an empty machine and takes it back when the session ends,
# so this installs the packages the tutorial needs. Run it again whenever
# you reopen the notebook.

import importlib
import importlib.util
import subprocess
import sys

REQUIRED = {                      # import name -> pip name
    "cartopy": "cartopy",
    "h5netcdf": "h5netcdf",
    "h5py": "h5py",
    "matplotlib": "matplotlib",
    "netCDF4": "netCDF4",
    "numpy": "numpy",
    "requests": "requests",
    "scipy": "scipy",
    "xarray": "xarray",
}

missing = [spec for mod, spec in REQUIRED.items()
           if importlib.util.find_spec(mod) is None]

if missing:
    print("Installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing],
                   check=False)
    importlib.invalidate_caches()

    # Check they import, rather than that pip said it was fine: a package can
    # install cleanly and still fail to load, and the fix for that is a
    # restart, which pip cannot do for you.
    broken = []
    for mod in REQUIRED:
        try:
            importlib.import_module(mod)
        except Exception as error:        # noqa: BLE001 - report anything
            broken.append(f"{mod} ({type(error).__name__}: {error})")

    if broken:
        print("\nInstalled, but these still will not import:")
        for line in broken:
            print("   ", line)
        print("Runtime -> Restart session, then run this cell again.")
    else:
        print("Ready.")
else:
    print("Ready - nothing to install.")

In [ ]:
# --- Folders and the figure saver: run this after the setup cell above. ---
# The setup cell installed the packages. This one makes the folders the
# notebook writes into, and defines save_fig(). Safe to re-run.
import os
import warnings

# Downloads go in data/, figures in output/, both beside this notebook.
SWOT_DIR = os.path.join(os.getcwd(), "data", "SWOT")
OSCAR_DIR = os.path.join(os.getcwd(), "data", "OSCAR")
FIG_DIR = os.path.join(os.getcwd(), "output", "figures")
for d in (SWOT_DIR, OSCAR_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

# Silence one harmless shapely warning that cartopy triggers on every map.
warnings.filterwarnings(
    "ignore", category=RuntimeWarning,
    message="invalid value encountered in create_collection")

# Turn off Jupyter's tight crop, which mangles cartopy maps into thin strips.
try:
    _shell = get_ipython()
except NameError:                      # not running under IPython
    _shell = None
if _shell is not None:
    _shell.run_line_magic(
        "config", "InlineBackend.print_figure_kwargs = {'bbox_inches': None}")

# cartopy puts a title at an infinite y when an axes has labelled gridlines, so
# the title falls off the canvas and out of the saved PNG. Pin it to the top of
# the axes instead; harmless on ordinary plots.
import matplotlib as mpl

mpl.rcParams["axes.titley"] = 1.0
mpl.rcParams["axes.titlepad"] = 6


def save_fig(fig, name, dpi=200):
    """Write a figure to output/figures/ so you can put it in your Word document.

    On Colab, open the folder icon in the left sidebar, find the file, and
    use its three-dot menu to Download it. The runtime is thrown away when
    the session ends, so download what you need before you close the tab.
    """
    path = os.path.join(FIG_DIR, f"{name}.png")
    fig.savefig(path, dpi=dpi)
    print("Saved figure:", os.path.relpath(path, os.getcwd()))
    return path

print("SWOT files go in        :", SWOT_DIR)
print("OSCAR files go in       :", OSCAR_DIR)
print("Figures will be saved to:", FIG_DIR)

## 1. Getting the data

Run the cell below to download every extract, August 2026 included.

Either container works, the loader in section 2 reads both MATLAB `.mat` and NetCDF, and everything after that treats them identically.

The second cell lists what is now in the folder, so you can check you have what you expect before going on.

In [ ]:
import requests

# The processed files for this tutorial.
COURSE_DATA = ("https://eyes-on-earth-tutorial-resources.s3.ap-southeast-1"
               ".amazonaws.com/tutorial_2/processed_data")


def download(url, dest_dir):
    """Download one file, skipping it if already there.

    Writes to a .part file first, so an interrupted download leaves nothing
    truncated behind.
    """
    name = url.split("/")[-1]
    dest = os.path.join(dest_dir, name)
    if os.path.exists(dest):
        print(f"Already have {name}")
        return dest

    tmp = dest + ".part"
    print(f"Downloading {name} ...", end=" ", flush=True)
    try:
        with requests.get(url, stream=True, timeout=600) as r:
            if r.status_code in (401, 403):
                raise PermissionError(
                    f"{r.status_code} from the course bucket for {name}. The "
                    "file is there but not readable without credentials - tell "
                    "your instructor, or get the file from them directly.")
            r.raise_for_status()
            with open(tmp, "wb") as fh:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    fh.write(chunk)
        os.replace(tmp, dest)
    except BaseException:
        if os.path.exists(tmp):
            os.remove(tmp)
        raise

    print(f"done ({os.path.getsize(dest) / 1e6:.0f} MB)")
    return dest

# Pre-processed by your instructor, so you do not have to reduce hundreds of
# SWOT passes yourself.
SSHA_FILES = [
    "SWOT_SSHA_202312.mat",     # December 2023
    "SWOT_SSHA_202407.mat",     # July 2024
    "SWOT_SSHA_202608.nc",      # August 2026, the current El Nino, for question 8
]

for name in SSHA_FILES:
    download(f"{COURSE_DATA}/SWOT_SSHA/{name}", SWOT_DIR)

In [ ]:
import glob

# Either container works - the loader in section 2 handles both.
paths = sorted(glob.glob(os.path.join(SWOT_DIR, "*.nc")) +
               glob.glob(os.path.join(SWOT_DIR, "*.mat")))
print(f"{len(paths)} file(s) found:")
for p in paths:
    print(" ", os.path.basename(p))

## 2. Why this data looks different

### Grids and point clouds

Notebooks 2.1 and 2.2 gave you **grids**, one value per cell. These SWOT extracts are the step before that: a **list of samples**, one row per measurement, in the order the satellite collected them.

| Column | |
|---|---|
| 1 | Longitude |
| 2 | Latitude |
| 3 | SSHA, in metres |

Two consequences shape the rest of part 1:

1. **There is no `ssha[lat, lon]` array**, so `sel(lon=slice(...))` has nothing to slice.
2. **Section 3 scatters one dot per sample** instead of using `pcolormesh`, which needs a mesh.

Gaps in your map are therefore not cloudy pixels, they are **places the satellite did not fly over**.

### What the number means

A radar altimeter times a pulse down to the water and back, which gives the height of the surface. Subtract the long-term average of that height — the **mean sea surface** — and the **anomaly** left over tracks the **heat stored in the column below**:

> **10 cm of SSHA ≈ 30 m of thermocline depth.**

The **thermocline** is the sharp boundary between the warm surface layer and the cold water beneath it, so its depth is the thickness of the warm layer. You use that rule in the assignment. An SSHA map is a map of where the warm water sits, which is what ENSO moves around.

### What the loader does

`load_ssha()` reads either container, MATLAB `.mat` or NetCDF, drops samples with no valid anomaly, and returns three plain NumPy arrays.

In [ ]:
import numpy as np
import xarray as xr


def load_ssha(path):
    """Return (lon, lat, ssha) as three 1-D arrays, with the gaps dropped.

    SSHA is in metres. Points where the anomaly is not finite are the gaps
    between swaths and in the nadir gap - there is nothing to plot there.
    """
    if path.endswith(".mat"):
        import scipy.io

        try:
            raw = scipy.io.loadmat(path, squeeze_me=True)
            data = np.asarray(raw["master_data"], dtype="float64")
        except NotImplementedError:
            # MATLAB v7.3 is really HDF5, which loadmat cannot read. h5py can,
            # and stores the array transposed relative to older versions.
            import h5py

            with h5py.File(path, "r") as f:
                data = np.array(f["master_data"]).T
    else:
        with xr.open_dataset(path) as d:
            data = np.column_stack([d["lon"].values.ravel(),
                                    d["lat"].values.ravel(),
                                    d["ssha"].values.ravel()])

    lon, lat, ssha = data[:, 0], data[:, 1], data[:, 2]
    keep = np.isfinite(ssha)
    return lon[keep], lat[keep], ssha[keep]

lon, lat, ssha = load_ssha(paths[0])

In [ ]:
print("File        :", os.path.basename(paths[0]))
print("Samples     :", f"{ssha.size:,}")
print("Lon range   :", f"{lon.min():.2f} to {lon.max():.2f}")
print("Lat range   :", f"{lat.min():.2f} to {lat.max():.2f}")
print("SSHA range  :", f"{ssha.min():.3f} to {ssha.max():.3f} m")
print("SSHA max|.| :", f"{np.abs(ssha).max():.3f} m   <- useful for setting the colour limits")

## 3. Our first SSHA map

Three choices are built into the cell.

**A diverging colour map (`RdBu_r`) with limits symmetric about zero.** SSHA is an anomaly, so the sign is the point: red is water standing high with warm water piled up, blue the opposite.

**`EXTENT` is in the projection's coordinates, not degrees east.** The region crosses the antimeridian, so the projection is centred on 180°. In that frame 180°E is `0`, 90°E is `-90` and 60°W is `+120`.

**`VLIM` is fixed rather than fitted to each file**, so the periods stay comparable.

That last one matters, because the sea surface varies on two scales at once and **which one you see depends on `VLIM`**:

- **Basin scale, which is ENSO.** A tilt across the whole equatorial Pacific, tens of centimetres end to end, changing over months. Questions 4–8 are about this.
- **Mesoscale, which is eddies.** Swirls 100–300 km across, also tens of centimetres high, gone within months. Mapping them is one of the main reasons SWOT was built.

Wide limits average the eddies into a smooth tilt; tight limits saturate the tilt and make the eddies stand out.

### Settings you can change

`VLIM` is the half-width of the colour scale in metres, `POINT_SIZE` the dot size. Run it at `1.0`, then `0.3`, then `0.1`, and settle on one to use for the rest of the notebook.

### 🔍 What to notice

- At which `VLIM` is the east–west tilt obvious? At which are the eddies?
- The **swath structure**: two 50 km swaths either side of a 20 km nadir gap.
- The diamond pattern where ascending and descending passes cross.

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

# ── Try changing these ────────────────────────────────────────────────────
VLIM = 0.3          # metres, half-width of the colour scale - try 1.0, 0.3, 0.1
POINT_SIZE = 2      # size of each sample's dot
# ─────────────────────────────────────────────────────────────────────────

PACIFIC = ccrs.PlateCarree(central_longitude=180)
EXTENT = [-90, 120, -30, 30]     # 90E to 60W, in the projection's own frame


def plot_ssha(lon, lat, ssha, title, vlim=None, size=POINT_SIZE):
    """Scatter one file's SSHA samples over the tropical Pacific."""
    fig, ax = plt.subplots(figsize=(13, 5), subplot_kw={"projection": PACIFIC})
    ax.set_extent(EXTENT, crs=PACIFIC)

    # One dot per sample. Raise `size` if the swaths look sparse, lower it if
    # they smear together.
    vlim = VLIM if vlim is None else vlim
    im = ax.scatter(lon, lat, c=ssha, s=size, marker="o", linewidths=0,
                    transform=ccrs.PlateCarree(),
                    cmap="RdBu_r", vmin=-vlim, vmax=vlim)

    ax.coastlines(linewidth=0.5)
    ax.add_feature(cfeature.LAND, facecolor="0.85", zorder=2)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="0.6")
    gl.top_labels = gl.right_labels = False
    ax.set_title(title, fontsize=12)

    fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02,
                 label="Sea surface height anomaly (m)")
    return fig, ax


p1, p50, p99 = np.percentile(ssha, [1, 50, 99])
print(f"{os.path.basename(paths[0])}: {ssha.size:,} samples, "
      f"1st percentile {p1:+.3f} m, median {p50:+.3f} m, 99th percentile {p99:+.3f} m, "
      f"standard deviation {ssha.std():.3f} m")
print(f"With VLIM = {VLIM}: {100 * np.mean(np.abs(ssha) > VLIM):.1f}% of samples are saturated.")

fig, ax = plot_ssha(lon, lat, ssha,
                    f"SWOT SSHA - {os.path.basename(paths[0])}, colour limits ±{VLIM} m")
save_fig(fig, f"ssha_first_file_vlim{VLIM:g}")
plt.show()

## 4. The same map for every period

Now run the same plot for every file you have, using the `VLIM` you settled on in section 3.

The loop below deliberately uses one `vlim` for all of them. This is not a cosmetic choice: the assignment asks you to compare periods, and a comparison between maps drawn on different scales is not a comparison at all.

In [ ]:
for path in paths:
    stem = os.path.splitext(os.path.basename(path))[0]
    p_lon, p_lat, p_ssha = load_ssha(path)
    fig, ax = plot_ssha(p_lon, p_lat, p_ssha,
                        f"SWOT SSHA - {os.path.basename(path)}", vlim=VLIM)
    save_fig(fig, f"ssha_{stem}_vlim{VLIM:g}")
    plt.show()

Every figure above has been saved to `output/figures/` as a PNG. On Colab, open that folder with the folder icon in the left sidebar, then use a file's three-dot menu and choose **Download**, and insert it into your Word document.

If you make any other figure of your own, pass it to the same helper to save it:

```python
fig, ax = plot_ssha(lon, lat, ssha, "SWOT SSHA — December 2023")
save_fig(fig, "ssha_dec2023")
```

## 5. Watch a month being collected *(optional)*

Section 3 drew a whole month of samples at once. This cell shows how that month got built, one day at a time: it reads the day each sample was measured, draws one frame per day, everything so far in faint colour, that day's passes bright on top, and saves a GIF to `output/figures/`.

### 🔍 What to look for

SWOT repeats its ground track every **21 days**. Watch for the day when new passes stop landing in empty water and start retracing tracks you have already seen. That moment is the repeat cycle, measured off your own data rather than read off a spec sheet, and a good check on the answer you gave for SWOT's repeat cycle.

> **Only the August 2026 file has the `day` column.** The December 2023 and July 2024 extracts were built before it was added, so the cell says so and stops rather than failing confusingly.

In [ ]:
# ── Try changing these ────────────────────────────────────────────────────
ANIM_FILE = "SWOT_SSHA_202608.nc"   # which file to animate
ANIM_POINTS = 120_000             # samples drawn, thinned so the frames render quickly
FPS = 3                           # frames per second in the GIF
# ─────────────────────────────────────────────────────────────────────────

import warnings

import matplotlib.animation as manim
from IPython.display import HTML

anim_path = os.path.join(SWOT_DIR, ANIM_FILE)
if not os.path.exists(anim_path):
    raise FileNotFoundError(
        f"{ANIM_FILE} is not in {SWOT_DIR}. Run section 1 first.")

with xr.open_dataset(anim_path) as d:
    if "day" not in d:
        raise ValueError(
            f"{ANIM_FILE} has no `day` variable, so there is nothing to animate. "
            "The December 2023 and July 2024 extracts predate it. Ask your "
            "instructor for an extract that has it.")
    a_lon = d["lon"].values
    a_lat = d["lat"].values
    a_ssha = d["ssha"].values
    a_day = d["day"].values

# Thinned for drawing only. Every frame redraws every sample collected so far,
# so the last frame carries the whole month.
step = max(1, a_ssha.size // ANIM_POINTS)
a_lon, a_lat, a_ssha, a_day = (v[::step] for v in (a_lon, a_lat, a_ssha, a_day))
days = np.arange(1, int(a_day.max()) + 1)
print(f"{a_ssha.size:,} samples over {len(days)} days (1 in {step} drawn)")

fig, ax = plt.subplots(figsize=(12, 4.6), subplot_kw={"projection": PACIFIC})
ax.set_extent(EXTENT, crs=PACIFIC)
ax.coastlines(linewidth=0.5)
ax.add_feature(cfeature.LAND, facecolor="0.85", zorder=2)
gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="0.6")
gl.top_labels = gl.right_labels = False

# Two layers: everything already collected, faint; today's passes, bright.
style = dict(cmap="RdBu_r", vmin=-VLIM, vmax=VLIM,
             transform=ccrs.PlateCarree(), linewidths=0)
before = ax.scatter([0], [0], c=[0.0], s=1.0, alpha=0.5, **style)
today = ax.scatter([0], [0], c=[0.0], s=5.0, **style)
fig.colorbar(today, ax=ax, shrink=0.8, pad=0.02,
             label="Sea surface height anomaly (m)")
# Place the title explicitly, or cartopy's gridlines push it off the canvas.
heading = ax.set_title("", fontsize=11, y=1.0, pad=6)

# 1 degree cells, only to put a coverage number in the title. Counts land too.
lon_edges, lat_edges = np.arange(-90, 121, 1), np.arange(-30, 31, 1)
proj_lon = (a_lon % 360) - 180        # into the projection's frame, as elsewhere


def draw(day):
    past, now = a_day < day, a_day == day
    before.set_offsets(np.c_[a_lon[past], a_lat[past]])
    before.set_array(a_ssha[past])
    today.set_offsets(np.c_[a_lon[now], a_lat[now]])
    today.set_array(a_ssha[now])

    seen = a_day <= day
    grid, _, _ = np.histogram2d(proj_lon[seen], a_lat[seen],
                                bins=[lon_edges, lat_edges])
    heading.set_text(f"SWOT SSHA collection — day {day} of {ANIM_FILE[-9:-3]}:  "
                     f"{int(seen.sum()):,} samples, "
                     f"{100 * (grid > 0).mean():.0f}% of 1° cells touched")
    return before, today, heading

movie = manim.FuncAnimation(fig, draw, frames=days,
                            interval=1000 / FPS, blit=False)

gif_path = os.path.join(FIG_DIR, f"ssha_collection_{ANIM_FILE[-9:-3]}.gif")
# Silence the same harmless shapely warning while the frames render.
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="invalid value encountered in create_collection")
    movie.save(gif_path, writer=manim.PillowWriter(fps=FPS), dpi=80)
print(f"Saved animation: {os.path.relpath(gif_path, os.getcwd())} "
      f"({os.path.getsize(gif_path) / 1e6:.1f} MB)")
plt.close(fig)

HTML(movie.to_jshtml())

## 6. The tilt of the equatorial Pacific *(required)*

**Assignment questions 5 and 8 come from this cell.**

Trade winds blow west along the equator and pile warm water into the western Pacific, so the sea surface genuinely slopes: **about half a metre higher near Indonesia than off South America**. The thermocline slopes the *other* way, some 150 m deep in the west and tens of metres in the east.

SSHA shows changes *from* that mean state. When the trade winds weaken in an **El Niño**, water held in the west sloshes east: SSHA goes positive in the east, negative in the west, and the thermocline flattens. **La Niña** does the opposite.

So **east-minus-west SSHA along the equator is a one-number index of the basin**, and its sign tells you which way the warm water moved.

The samples are a point cloud, not a grid, so before anything can be averaged along the equator or subtracted between periods they have to be **binned**: lay down cells `BIN_DEG` degrees across and average every sample that fell inside each one. Some cells collect many more samples than others, so an average inherits that unevenness.

### Settings you can change

- `BIN_DEG`, the size in degrees of the cells the samples are averaged into. Try `0.5`, `1` and `2`, and watch where the swath gaps close and whether the eddies survive it.
- `LAT_BAND`, half-width of the equatorial strip averaged.
- `DIFF_ENSO` and `DIFF_NOW`, the two subtractions, each `(later, earlier)`. One serves question 5, the other question 8, and both are drawn every run.
- `WEST` and `EAST`, the boxes compared.

### 🔍 What to notice

**In the profiles:** where each curve crosses zero, and the east-minus-west number printed for each. The cell draws every file you downloaded: `SWOT_SSHA_202312` and `SWOT_SSHA_202407` for question 5, and `SWOT_SSHA_202608` for question 8. **In the difference maps:** where did the water go between each pair of periods?

In [ ]:
from scipy.stats import binned_statistic_2d


def grid_ssha(lon, lat, ssha, bin_deg):
    """Average scattered samples into `bin_deg` cells, in the projection's frame.

    This is the same binning that turns swath data into the Level-3 grids you
    used in 2.1 and 2.2. Small cells keep the detail and leave holes wherever no
    swath passed; big cells fill the holes and smooth the eddies away.
    """
    x = (lon % 360) - 180
    x_edges = np.arange(-90, 120 + bin_deg / 2, bin_deg)
    y_edges = np.arange(-30, 30 + bin_deg / 2, bin_deg)
    mean, _, _, _ = binned_statistic_2d(x, lat, ssha, statistic="mean", bins=[x_edges, y_edges])
    count, _, _, _ = binned_statistic_2d(x, lat, ssha, statistic="count", bins=[x_edges, y_edges])
    coords = {"lat": 0.5 * (y_edges[1:] + y_edges[:-1]), "lon": 0.5 * (x_edges[1:] + x_edges[:-1])}
    mean = xr.DataArray(mean.T, coords=coords, dims=("lat", "lon"), name="ssha", attrs={"units": "m"})
    count = xr.DataArray(count.T, coords=coords, dims=("lat", "lon"), name="samples")
    return mean, count.where(count > 0)


def draw_grid(da, ax, cmap, vmin, vmax):
    im = ax.pcolormesh(da.lon, da.lat, da, transform=PACIFIC, cmap=cmap,
                       vmin=vmin, vmax=vmax, shading="auto")
    ax.set_extent(EXTENT, crs=PACIFIC)
    ax.coastlines(linewidth=0.5)
    ax.add_feature(cfeature.LAND, facecolor="0.85", zorder=2)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="0.6")
    gl.top_labels = gl.right_labels = False
    return im


# ── Try changing these ────────────────────────────────────────────────────
BIN_DEG = 1.0                    # grid cell size in degrees - try 0.5, 1, 2
LAT_BAND = 2                     # degrees either side of the equator
# Two difference maps, one per assignment question, each given as
# (later, earlier). SWOT has no December 2022, so question 5 contrasts
# July 2024 with December 2023; for December 2022 use section 7.
DIFF_ENSO = ("202407", "202312")            # question 5: July 2024 minus December 2023
DIFF_NOW = ("202608", "202312")             # question 8: August 2026 minus December 2023
WEST, EAST = (140, 170), (210, 270)   # boxes in degrees east: 140-170E, and 150W-90W
# ─────────────────────────────────────────────────────────────────────────

grids = {}
for path in paths:
    stem = os.path.splitext(os.path.basename(path))[0]
    p_lon, p_lat, p_ssha = load_ssha(path)
    grids[stem], _ = grid_ssha(p_lon, p_lat, p_ssha, BIN_DEG)
stems = list(grids)


def stem_for(period):
    """Find the loaded period whose filename contains `period`."""
    hits = [s for s in stems if period.split("_")[-1].split(".")[0] in s]
    if not hits:
        raise KeyError(f"No file matching '{period}'. Section 1 loaded: "
                       + ", ".join(stems))
    return hits[0]

fig, ax = plt.subplots(figsize=(11, 4.5))
print(f"Equatorial SSHA ({LAT_BAND}S-{LAT_BAND}N), from {BIN_DEG} degree cells:")
for stem, g in grids.items():
    strip = g.sel(lat=slice(-LAT_BAND, LAT_BAND)).mean(dim="lat", skipna=True)
    lon_east = strip.lon.values + 180
    ax.plot(lon_east, strip.values, linewidth=1.6, label=stem)
    west = float(strip.sel(lon=slice(WEST[0] - 180, WEST[1] - 180)).mean())
    east = float(strip.sel(lon=slice(EAST[0] - 180, EAST[1] - 180)).mean())
    print(f"  {stem}: west box {west:+.3f} m, east box {east:+.3f} m, east minus west {east - west:+.3f} m")

ax.axhline(0, color="0.3", linewidth=0.8)
for box, name in ((WEST, "west box"), (EAST, "east box")):
    ax.axvspan(*box, color="0.9", zorder=0)
    ax.text(np.mean(box), ax.get_ylim()[1] * 0.9, name, ha="center", fontsize=8, color="0.4")
ax.set_xlabel("longitude (degrees east; 180 to 300 is 180°W to 60°W)")
ax.set_ylabel("SSHA (m)")
ax.set_xlim(90, 300)
ax.set_title(f"Equatorial SSHA, averaged over {LAT_BAND}°S–{LAT_BAND}°N", fontsize=11)
ax.grid(alpha=0.3)
ax.legend(fontsize=9)
save_fig(fig, f"ssha_equatorial_profile_eq{LAT_BAND}_{BIN_DEG:g}deg")
plt.show()

# One difference map per question, cell by cell. Both are drawn every run, so
# neither question depends on you having changed a setting for the other.
for question, pair in (("question 5", DIFF_ENSO), ("question 8", DIFF_NOW)):
    later, earlier = stem_for(pair[0]), stem_for(pair[1])
    diff = grids[later] - grids[earlier]
    fig, ax = plt.subplots(figsize=(13, 5), subplot_kw={"projection": PACIFIC})
    im = draw_grid(diff, ax, "RdBu_r", -VLIM, VLIM)
    ax.set_title(f"SSHA difference for {question}: {later} minus {earlier}  "
                 f"({BIN_DEG}° cells)", fontsize=11)
    fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02, label="difference in SSHA (m)")
    band = diff.sel(lat=slice(-LAT_BAND, LAT_BAND))
    print(f"\n{question}: {later} minus {earlier}, {LAT_BAND}S-{LAT_BAND}N: "
          f"west box {float(band.sel(lon=slice(WEST[0] - 180, WEST[1] - 180)).mean()):+.3f} m, "
          f"east box {float(band.sel(lon=slice(EAST[0] - 180, EAST[1] - 180)).mean()):+.3f} m")
    save_fig(fig, f"ssha_diff_{later}_minus_{earlier}_{BIN_DEG:g}deg")
    plt.show()

> ### 📝 For your Word document (2.3 §6 — required)
>
> Keep the equatorial profile and both difference maps, and the printed west-box, east-box and east-minus-west numbers.
>
> **The cell draws two difference maps, and the title of each says which question it is for.** `DIFF_ENSO` gives July 2024 minus December 2023 for question 5; `DIFF_NOW` gives August 2026 minus December 2023 for question 8. Both come out of one run, you do not have to change a setting to get either.
>
> [Assignment questions 5 and 8](https://github.com/earthobservatory/eyes-on-earth-tutorials-public/blob/main/2.0_Tutorial_2_Overview_and_Assignment/README.md#part-b--analyse-the-ocean) in the handout say what to write.

## 7. December 2022 — the period SWOT cannot help with

SWOT launched on **16 December 2022** and spent the following months in commissioning. The earliest science-quality SSHA is from **27 March 2023**, so there is no December 2022 SWOT map to make.

The assignment still asks about December 2022, because it is one of the three ENSO phases. Use the composite below instead: **Sentinel-6 and Jason-3**, conventional nadir altimeters rather than SWOT's wide-swath interferometer, averaged over 10–20 December 2022.

![Sea surface height anomaly, 10-20 December 2022, from Sentinel-6 and Jason-3](https://raw.githubusercontent.com/earthobservatory/eyes-on-earth-tutorials-public/main/2.3_Sea_Surface_Height_and_Ocean_Currents/Figs/ssha_composite_dec2022.png)

### 🔍 Two things to notice when you compare it with your own maps

- **The units are centimetres here**, while the SWOT files are in metres. Do not read the two colour scales as though they were the same.
- **The coverage is visibly striped along satellite tracks.** A nadir altimeter measures a single line directly beneath itself; SWOT measures two 50 km swaths. That striping is the limitation SWOT was built to address, worth a sentence in mission question 1.


---

# Part 2 — Ocean surface currents

Sections 1 to 7 measured **where the water is piled up**. Sections 8 to 13 measure **which way it is moving**, and the two are the same story told twice: away from the equator, a current runs along the contours of that height field, so a map of the slope is very nearly a map of the flow.

Everything below uses a different satellite product, **OSCAR**, and its own download. Nothing above needs re-running.

## 8. Nothing in orbit measures a current

PACE measures colour. Himawari measures infrared radiation. SWOT measures the travel time of a radar pulse. **OSCAR measures none of it.** It *calculates* surface currents from things satellites do measure: sea surface height, wind and temperature. What you map here is a model output constrained by observations, not an observation.

That is the thing to remember about this dataset, because it tells you which parts to trust. The broad current bands, their direction, and how they change between the three periods are what the calculation is good at. A single arrow, or a feature one or two grid cells across, is interpolation.

Two pieces of physics go into it: **geostrophic flow**, which OSCAR reads off the slope of the sea surface you mapped in part 1, and **Ekman flow**, driven by the wind. The geostrophic balance breaks down on the equator, where OSCAR substitutes a different approximation, so treat the equatorial band as the least certain part of the map.

### What is in the file

| Variable | Meaning |
|---|---|
| `u`, `v` | **Total** surface current, eastward and northward components |
| `ug`, `vg` | **Geostrophic** surface current, eastward and northward components |

The assignment asks for the **total** current, so the maps use `u` and `v`. Velocities are an average over the top 30 m, in m s⁻¹, on a 0.25° grid.

## 9. Getting the current data

Run the cell below: one file per time period.

> The December 2022 file is dated the **10th** rather than the 1st. These are daily near-real-time fields, one date standing in for each period, not monthly means like the PACE and Himawari products.

You can also fetch them from PO.DAAC with `earthaccess` (`pip install earthaccess` first, on Colab), which reads your Earthdata credentials from `~/.netrc`, set up in [0.2.1 Account Check](https://github.com/earthobservatory/eyes-on-earth-tutorials-public/blob/main/0.2_Data_Access_Accounts/0.2.1_Account_Check.ipynb):

```python
import earthaccess

auth = earthaccess.login(strategy="netrc")
results = earthaccess.search_data(
    short_name="OSCAR_L4_OC_NRT_V2.0",
    temporal=("2023-12-01", "2023-12-02"),
)
earthaccess.download(results, OSCAR_DIR)
```

The second cell lists what you have and prints the structure of the first file. **Look closely at the dimensions**, they are not laid out the way the previous notebooks were, and section 10 has to sort it out.

In [ ]:
# download() and COURSE_DATA come from section 1, which you have already run.
OSCAR_FILES = [
    "oscar_currents_nrt_20221210.nc",   # 10 December 2022
    "oscar_currents_nrt_20231201.nc",   # 1 December 2023
    "oscar_currents_nrt_20240701.nc",   # 1 July 2024
]

for name in OSCAR_FILES:
    download(f"{COURSE_DATA}/OSCAR_ocean_currents/{name}", OSCAR_DIR)

In [ ]:
import glob

import xarray as xr

oscar_paths = sorted(glob.glob(os.path.join(OSCAR_DIR, "*.nc")))
print(f"{len(oscar_paths)} file(s):")
for p in oscar_paths:
    print(" ", os.path.basename(p))

# The dimensions are named longitude/latitude while the coordinates are
# lon/lat. Section 10 sorts that out.
ds = xr.open_dataset(oscar_paths[0])
ds

## 10. From two components to a speed and a direction

`u` and `v` are two halves of one vector: `u` is how fast the water moves **eastward**, `v` how fast **northward**. To show a current you need both, and they go onto the map differently: the **speed**, `sqrt(u² + v²)`, as a colour field, and the **direction** as arrows on top.

The arrows get two adjustments inside `plot_currents()`, both of which section 11 lets you change:

1. **Thinned out.** One arrow per grid point would cover the map in solid white, so the cell draws every 12th point.
2. **All the same length.** The colour field already carries the speed, so the arrows only carry direction. Left proportional to speed, the slow equatorial currents shrink until their direction cannot be read, and those are where the El Niño signal is.

**Two things the loader fixes.** The **dimensions** are named `longitude` and `latitude` while the **coordinate variables** are named `lon` and `lat`, and those are not automatically connected, so `.sel(lat=...)` does not work until `swap_dims` links them. And the arrays arrive as `(longitude, latitude)`, so they are transposed into the `(lat, lon)` order the rest of the tutorial assumes.

In [ ]:
import numpy as np


def load_currents(path, variables=("u", "v")):
    """Read total current components, with lat/lon as usable coordinates.

    Two things need fixing on the way in. The dimensions in these files are
    named `longitude`/`latitude` while the coordinate variables are `lon`/`lat`,
    so `swap_dims` makes each coordinate the index of its own dimension - after
    which `.sel(lat=...)` by degrees works. And the arrays arrive as
    (longitude, latitude), so they are transposed to the (lat, lon) order that
    `pcolormesh` and the rest of this tutorial assume.
    """
    with xr.open_dataset(path) as d:
        out = []
        for name in variables:
            da = d[name].squeeze(drop=True)
            da = da.swap_dims({"longitude": "lon", "latitude": "lat"})
            out.append(da.transpose("lat", "lon").load())
    return out


def date_label(path):
    """'oscar_currents_nrt_20221210.nc' -> '10 Dec 2022'."""
    from datetime import datetime
    stamp = os.path.basename(path).split("_")[3][:8]
    return datetime.strptime(stamp, "%Y%m%d").strftime("%d %b %Y")

u, v = load_currents(oscar_paths[0])

speed = np.sqrt(u ** 2 + v ** 2)
speed.attrs["units"] = "m s-1"
speed.name = "total current speed"

print("Grid dims  :", u.dims, u.shape)
print("Lat range  :", float(u.lat.min()), "to", float(u.lat.max()))
print("Lon range  :", float(u.lon.min()), "to", float(u.lon.max()))
print("Speed range:", f"{float(speed.min()):.3f} to {float(speed.max()):.3f} m/s")

## 11. Mapping the total surface currents

Our region is the tropical Pacific, **30°S–30°N, 90°E–60°W**, the same box as part 1 and notebook 2.2.

As in part 1, the region crosses the antimeridian, so the projection is centred on 180° and `EXTENT` is given in the projection's own coordinates, not degrees east (section 3 has the conversion).

**Before you run it, predict something.** You know from part 1 of this notebook that the sea surface is higher in the west and lower in the east. You know from section 8 that water flows *along* a slope, not down it. So which way do you expect the water on the equator to be moving, east or west? And what happens to that reasoning exactly on the equator, where the Coriolis parameter is zero?

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

# The region crosses the antimeridian, so the projection is centred on the
# Pacific and EXTENT is in its frame: 180E is 0, 90E is -90, 60W is +120.
PACIFIC = ccrs.PlateCarree(central_longitude=180)
EXTENT = [-90, 120, -30, 30]
STEP = 12                           # draw one arrow every STEP grid points


def plot_currents(u, v, title, vmax=1.0, step=STEP, normalise=True):
    speed = np.sqrt(u ** 2 + v ** 2)

    fig, ax = plt.subplots(figsize=(13, 5), subplot_kw={"projection": PACIFIC})
    ax.set_extent(EXTENT, crs=PACIFIC)

    im = ax.pcolormesh(u.lon, u.lat, speed, transform=ccrs.PlateCarree(),
                       cmap="turbo", vmin=0, vmax=vmax, shading="auto")

    # Thin the arrows, then normalise to unit length so they show direction
    # only. The guard avoids dividing by zero where the current is still.
    sub = dict(lon=slice(None, None, step), lat=slice(None, None, step))
    us, vs = u.isel(**sub), v.isel(**sub)
    if normalise:
        mag = np.sqrt(us ** 2 + vs ** 2)
        us, vs = (us / mag).where(mag > 0), (vs / mag).where(mag > 0)
        scale = 70                       # unit arrows: 70 of them across the axis
    else:
        scale = 70 * vmax                # raw m/s: an arrow of `vmax` is the same length

    ax.quiver(us.lon, us.lat, us.values, vs.values,
              transform=ccrs.PlateCarree(), scale=scale, width=0.0015,
              color="white", alpha=0.9)

    ax.coastlines(linewidth=0.5)
    ax.add_feature(cfeature.LAND, facecolor="0.85", zorder=2)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="0.6")
    gl.top_labels = gl.right_labels = False
    ax.set_title(title, fontsize=12)

    fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02,
                 label="Total surface current speed (m s$^{-1}$)")
    return fig, ax

fig, ax = plot_currents(u, v,
                        f"OSCAR total surface currents — {os.path.basename(oscar_paths[0])}")
save_fig(fig, "currents_first_file")
plt.show()

## 12. All three time periods *(required)*

**The three maps this cell draws answer assignment question 6.** Look at all three; the handout asks you to paste in only the December 2023 one.

The cell below draws all three periods with the same colour limits, `vmax = 1.0` m s⁻¹ for each. Limits fitted to each period separately would let every map use the full range of colours, and all three would end up looking much alike.

OSCAR has all three periods, December 2022 included. That is the period SWOT could not give you in part 1.

Run it, then compare the flow along the equator between the three. Section 13 measures what you are looking at.

In [ ]:
for path in oscar_paths:
    stem = os.path.splitext(os.path.basename(path))[0]
    p_u, p_v = load_currents(path)
    fig, ax = plot_currents(
        p_u, p_v,
        f"OSCAR total surface currents — {date_label(path)} ({os.path.basename(path)})",
        vmax=1.0,
    )
    save_fig(fig, f"currents_{stem}")
    plt.show()

## 13. Which way is the water moving on the equator? *(required)*

**Assignment question 6 uses the numbers this cell prints, alongside the maps from section 12.**

The maps above show speed and direction everywhere. For the assignment you need one number per period: how fast the water is moving *along* the equator, and which way. This cell measures it two ways.

The left panel averages the eastward current over a range of longitudes and plots it against latitude. The right panel averages over a narrow strip either side of the equator and plots it against longitude.

The tropical Pacific sorts itself into west-flowing and east-flowing stripes, and the shaded bands on the left panel name the three largest. You do not need to learn those names. What matters for ENSO is the sign on the equator itself. The trade winds push surface water west, so the equatorial flow is normally westward. When the trades weaken in an **El Niño** that westward flow weakens, and can turn eastward, letting warm water drain back towards the east — the same movement you measured as an SSHA change in section 6. In a **La Niña** the trades are strong and the westward flow is at its strongest.

### Settings you can change

- `LON_RANGE`, the band of longitudes averaged for the latitude profile. Try `(140, 170)` for the western warm pool and `(210, 240)` for the east.
- `LAT_BAND`, half-width of the strip either side of the equator used for the second plot.

### 🔍 What to notice

Which period has eastward flow on the equator, and where along the equator the flow is westward and where it is not.

**Sign convention: positive is eastward, negative is westward.**

In [ ]:
# ── Try changing these ────────────────────────────────────────────────────
LON_RANGE = (160, 200)      # degrees east, averaged for the latitude profile
LAT_BAND = 2                # degrees either side of the equator, for the longitude profile
# ─────────────────────────────────────────────────────────────────────────

BANDS = [("NEC", 10, 20, "westward"), ("NECC", 4, 9, "eastward"), ("SEC", -10, 3, "westward")]

fig, (ax_lat, ax_lon) = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [1, 1.6]})

print(f"Zonal-mean u over {LON_RANGE[0]}E-{LON_RANGE[1]}E:")
for path in oscar_paths:
    label = date_label(path)
    (p_u,) = load_currents(path, variables=("u",))

    # Against latitude, averaged over LON_RANGE.
    by_lat = p_u.sel(lon=slice(*LON_RANGE), lat=slice(-20, 20)).mean(dim="lon", skipna=True)
    ax_lat.plot(by_lat.values, by_lat.lat, linewidth=1.6, label=label)

    # Along the equator, averaged over the strip.
    band = p_u.sel(lat=slice(-LAT_BAND, LAT_BAND))
    if band.sizes["lat"] == 0:                    # in case lat runs north-to-south
        band = p_u.sel(lat=slice(LAT_BAND, -LAT_BAND))
    ax_lon.plot(band.lon, band.mean(dim="lat"), label=label, linewidth=1.2)

    eq = float(by_lat.sel(lat=slice(-LAT_BAND, LAT_BAND)).mean())
    necc = by_lat.sel(lat=slice(4, 9))
    print(f"  {label}: on the equator {eq:+.2f} m/s; strongest eastward flow in 4-9N "
          f"{float(necc.max()):+.2f} m/s at {float(necc.lat[int(np.argmax(necc.values))]):.1f}N")

for name, south, north, direction in BANDS:
    ax_lat.axhspan(south, north, color="0.93", zorder=0)
    ax_lat.text(ax_lat.get_xlim()[0] + 0.02, (south + north) / 2, f"{name} ({direction})",
                fontsize=8, va="center", color="0.4")
ax_lat.axvline(0, color="0.3", linewidth=0.8)
ax_lat.set_xlabel("eastward speed u (m s$^{-1}$)")
ax_lat.set_ylabel("latitude")
ax_lat.set_title(f"Zonal-mean u, {LON_RANGE[0]}E-{LON_RANGE[1]}E", fontsize=11)
ax_lat.grid(alpha=0.3)
ax_lat.legend(fontsize=8, loc="lower right")

ax_lon.axhline(0, color="0.4", linewidth=0.8)
ax_lon.set_xlim(140, 285)          # west of 140E the strip runs through the Indonesian seas
ax_lon.set_xlabel("longitude (degrees east; 180 to 285 is 180°W to 75°W)")
ax_lon.set_ylabel("eastward speed u (m s$^{-1}$)")
ax_lon.set_title(f"u along the equator, averaged over {LAT_BAND}S-{LAT_BAND}N", fontsize=11)
ax_lon.legend(fontsize=8)
ax_lon.grid(alpha=0.3)

save_fig(fig, f"currents_bands_{LON_RANGE[0]}E_{LON_RANGE[1]}E_eq{LAT_BAND}")
plt.show()

> ### 📝 For your Word document (2.3 §12 and §13 — required)
>
> Keep the December 2023 total-current map from section 12 and the equatorial speeds section 13 prints. You will need the SSHA numbers from section 6 and your phase table from question 4 alongside them.
>
> [Assignment question 6](../2.0_Tutorial_2_Overview_and_Assignment/README.md#part-b--analyse-the-ocean) in the handout says what to write.

## Background reading

The links for SWOT, OSCAR and ENSO are collected under **Background reading** in the [module README](https://github.com/earthobservatory/eyes-on-earth-tutorials-public/blob/main/2.3_Sea_Surface_Height_and_Ocean_Currents/README.md), together with the ones the mission questions need.

## You have finished the notebooks

Now go back to the [tutorial README](https://github.com/earthobservatory/eyes-on-earth-tutorials-public/blob/main/2.0_Tutorial_2_Overview_and_Assignment/README.md) for the **assignment**, which asks you to bring chlorophyll-a, POC, SST, SSHA and surface currents together, and then to point them at an El Niño that is still going on.